## Importing Libraries

In [ ]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
from groq import Groq

## Setting up files

In [ ]:
GENERATION_MODEL = "qwen3:8b" 
GROQ_MODEL = "openai/gpt-oss-120b"

GROQ_KEY = os.getenv("GROQ_API_KEY")
CLIENT = Groq(api_key=GROQ_KEY)

TYPE_LLM = True # True - local, False - groq

FILES_ANALYSIS = glob.glob("../Test_Files/Analysis/analysis_patient_*.txt")
FILES_DIARIES = glob.glob("../Test_Files/Clinical_diaries/inconsistancy-diary_patient_*.txt")
FILE_RULES = "../Test_Files/Clinical_trials/Criteria_extracted/clinical-trial-extracted_e1.txt"

PROMPT_FILE = "./prompts/matching-patients/matching-patients_prompt.txt"
SYS_PROMPT_FILE = "./prompts/matching-patients/sys_matching-patients_prompt.txt"

OUTPUT_DIR = "./llm-outputs/matching-patients/"
OUTPUT_FILE = "experiment"

print(f"Found the following analysis - {FILES_ANALYSIS}")
print(f"Found the following diaries - {FILES_DIARIES}")
print(f"Found the following rules - {FILE_RULES}")

## Setting up environment

In [ ]:
## Setting evironment

with open(PROMPT_FILE,"r", encoding="utf-8") as p, open(SYS_PROMPT_FILE,"r", encoding="utf-8") as sp:
    base_prompt = p.read()
    sys_prompt = sp.read()

os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isfile(os.path.join(OUTPUT_DIR, path)):
        count += 1

## Justification generation
In this phase the justification generation for a eligibility decision will be done by a LLM, it must have the patient profile and the logic rule converted trial criteria for a clear justification

In [ ]:
def call_prompt(prompt, sys_prompt, file):

    if TYPE_LLM:
        stream = chat(
            model=GENERATION_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": sys_prompt
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            stream=True,
            options={"num_ctx": 32000}
        )

        llm_output = ""

        for chunk in stream:
            llm_output += chunk["message"]["content"]

    else:
        stream = CLIENT.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": sys_prompt
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        llm_output = stream.choices[0].message.content

    with open(
        f"{OUTPUT_DIR}{OUTPUT_FILE}-{count}.txt",
        "a",
        encoding="utf-8"
    ) as o:

        o.write(f"Output for file {file}\n")
        o.write(f"{llm_output}\n\n")

        print(f"Saved LLM output on {OUTPUT_FILE}-{count}")


total_criteria = 0

with open(FILE_RULES, 'r', encoding='utf-8') as trial_rules:
    data_rules = json.load(trial_rules)

    total_criteria = (
        len(FILES_DIARIES)
        * (
            len(data_rules["inclusion_criteria"])
            + len(data_rules["exclusion_criteria"])
        )
    )

pbar = tqdm(
    total=total_criteria,
    desc="Matching patients when there is an unknown schema field"
)


for diary in FILES_DIARIES:

    patient_id = diary.split("_")[-1].split(".")[0]

    patient_analysis = None

    for analysis in FILES_ANALYSIS:

        analysis_patient_id = analysis.split("_")[-1].split(".")[0]

        if analysis_patient_id == patient_id:

            print(f"Patient's analysis file - {analysis}")

            patient_analysis = analysis
            break

    if patient_analysis is None:
        print(f"No analysis found for patient {patient_id}")
        continue

    with open(diary, 'r', encoding='utf-8') as f, \
         open(patient_analysis, 'r', encoding='utf-8') as analysis_file, \
         open(FILE_RULES, 'r', encoding='utf-8') as trial_rules:

        data_rules = json.load(trial_rules)
        data_analysis = json.load(analysis_file)

        diary_content = f.read().strip()

        all_inclusion_criteria = data_rules["inclusion_criteria"]
        all_exclusion_criteria = data_rules["exclusion_criteria"]

        for criteria in all_inclusion_criteria:

            criteria_text = "INCLUSION CRITERION - " + criteria

            prompt_w_diary = base_prompt.replace(
                "{{CLINICAL_DIARY}}",
                diary_content
            )

            prompt_w_analysis = prompt_w_diary.replace(
                "{{ANALYSIS_VALUES}}",
                json.dumps(data_analysis)
            )

            prompt_final = prompt_w_analysis.replace(
                "{{CRITERION_TEXT}}",
                criteria_text
            )

            call_prompt(prompt_final, sys_prompt, diary)

            print("\n")

            pbar.update(1)

        for criteria in all_exclusion_criteria:

            criteria_text = "EXCLUSION CRITERION - " + criteria

            prompt_w_diary = base_prompt.replace(
                "{{CLINICAL_DIARY}}",
                diary_content
            )

            prompt_w_analysis = prompt_w_diary.replace(
                "{{ANALYSIS_VALUES}}",
                json.dumps(data_analysis)
            )

            prompt_final = prompt_w_analysis.replace(
                "{{CRITERION_TEXT}}",
                criteria_text
            )

            call_prompt(prompt_final, sys_prompt, diary)

            print("\n")

            pbar.update(1)

pbar.close()